In [2]:
from pathlib import Path
import shutil

dataset_dir = Path("../../../Downloads/dataset_repos")

print(dataset_dir.resolve())
print(dataset_dir.exists())

/Users/senamukai/Downloads/dataset_repos
True


In [3]:
#AGENTS.mdとCLAUDE.mdがデータセットファイル内に存在する総数
agents_files = list(dataset_dir.rglob("AGENTS.md"))
claude_files = list(dataset_dir.rglob("CLAUDE.md"))

print("AGENTS:", len(agents_files))
print("CLAUDE:", len(claude_files))

AGENTS: 3703
CLAUDE: 3395


In [4]:
#AGENTS.mdとCLAUDE.mdのそれぞれに対して，https://の総数を数えるコード
def total_count_https(files):
    count = 0

    for file in files: #GitHubリポジトリのCode上を見ている
        with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
            count += f.read().count("https://")
    return count

print("AGENTS:", total_count_https(agents_files))
print("CLAUDE:", total_count_https(claude_files))        


AGENTS: 4323
CLAUDE: 1833


In [5]:
"""
AGENTS.mdとCLAUDE.mdのhttps://を1つ以上含むファイルに対して，
1ファイルごとに1としてカウントし，ファイルフォルダーを作成するプログラム
"""
agents_output = Path("./https_AGENTS")
claude_output = Path("./https_CLAUDE")

agents_output.mkdir(exist_ok = True)
claude_output.mkdir(exist_ok = True)

#AGNTS.md files
agents_count = 0

for file in agents_files:

    with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
        text = f.read()

    if "https://" in text:
        relative_path = file.relative_to(dataset_dir)
        save_path = agents_output / relative_path
        save_path.parent.mkdir(parents = True, exist_ok = True)
        shutil.copy2(file, save_path)
        agents_count += 1

#CLAUDE.md files
claude_count = 0

for file in claude_files:

    with open(file, "r", encoding = "utf-8", errors = "ignore") as f:
        text = f.read()
        
    if "https://" in text:
        relative_path = file.relative_to(dataset_dir)
        save_path = claude_output / relative_path
        save_path.parent.mkdir(parents = True, exist_ok = True)
        shutil.copy2(file, save_path)
        claude_count += 1

print("https:// in AGENTS.md:", agents_count)
print("https:// in CLAUDE.md:", claude_count)

https:// in AGENTS.md: 917
https:// in CLAUDE.md: 635


In [ ]:
"""
dataset_repos内から，AGENTS.md or CLAUDE.mdのいずれかを含むリポジトリに対し，
10Mbyte以下のリポジトリを満たすか確認し，
どちらも満たした場合，抽出しそのリポジトリ丸ごとをwork2ディレクトリ下に配置するプログラム
"""

#元々のデータセット
dataset_dir = Path("../../../Downloads/dataset_repos")
output_dir = Path("../work2/under10Mbyte_AGENTS&&CLAUDE_repos")

#10Mbyteの計算
LIMIT_SIZE = 10 * 1024 * 1024

#出力フォルダ作成
output_dir.mkdir(parents = True, exist_ok = True)

#riposのサイズを求める関数
def get_dir_size(path):
    total = 0

    for p in path.rglob("*"):
        if p.is_file():
            total += p.stat().st_size

    return total   

#カウンタ変数の初期化
copied_repo_count = 0
skipped_no_agent_file = 0
skipped_over_10mb = 0

#riposを1個ずつ探索
for repo_dir in dataset_dir.iterdir():

    if not repo_dir.is_dir():
        continue

    #条件１: AGENTS.md or CLAUDE.md のいずれかを含むか確認
    has_agents = any(repo_dir.rglob("AGENTS.md"))
    has_claude = any(repo_dir.rglob("CLAUDE.md"))

    if not (has_agents or has_claude):
        skipped_no_agent_file += 1
        continue

    #条件2: リポジトリ全体のサイズが10Mbyte以下か確認
    repo_size = get_dir_size(repo_dir)

    if repo_size > LIMIT_SIZE:
        skipped_over_10mb += 1
        continue

    #条件1,2を満たす場合リポジトリをコピーする
    save_path = output_dir / repo_dir.name

    if save_path.exists():
        shutil.rmtree(save_path)

    shutil.copytree(repo_dir, save_path)
    copied_repo_count += 1

print("copied repositories:", copied_repo_count)
print("skipped no AGENTS.md or CLAUDE.md:", skipped_no_agent_file)
print("skipped over 10Mbyte:", skipped_over_10mb)
